# Execucao da mart de indicadores financeiros

Este notebook executa no PostgreSQL a query salva em `queries/03_gold/01_refaz_mart_indicadores_financeiros_todas_empresas.sql`.


## 1. Setup

O notebook:
- localiza a raiz do projeto
- carrega variaveis de ambiente do `.env`
- cria a conexao com PostgreSQL
- le a SQL do arquivo e executa no banco


In [1]:
from __future__ import annotations

import os
from pathlib import Path
from urllib.parse import quote_plus

from dotenv import load_dotenv
from sqlalchemy import create_engine, text


def find_repo_root() -> Path:
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        if (candidate / 'README.md').exists() and (candidate / 'queries').exists():
            return candidate
    raise FileNotFoundError('Nao foi possivel localizar a raiz do repositorio.')


ROOT = find_repo_root()
ENV_PATH = ROOT / '.env'
SQL_PATH = ROOT / 'queries' / '03_gold' / '01_refaz_mart_indicadores_financeiros_todas_empresas.sql'

load_dotenv(ENV_PATH)

DB_USER = quote_plus(os.getenv('DB_USER', 'postgres'))
DB_PASS = quote_plus(os.getenv('DB_PASS', 'password'))
DB_HOST = os.getenv('DB_HOST', 'localhost')
DB_PORT = os.getenv('DB_PORT', '5432')
DB_NAME = os.getenv('DB_NAME', 'data_lake')

engine = create_engine(f'postgresql+psycopg2://{DB_USER}:{DB_PASS}@{DB_HOST}:{DB_PORT}/{DB_NAME}')

print(f'Raiz do projeto: {ROOT}')
print(f'Arquivo SQL: {SQL_PATH}')
print(f'Banco alvo: {DB_HOST}:{DB_PORT}/{DB_NAME}')


Raiz do projeto: c:\Users\renan\Documents\FAE\Big Data for Finance\big_data_for_finance
Arquivo SQL: c:\Users\renan\Documents\FAE\Big Data for Finance\big_data_for_finance\queries\03_gold\01_refaz_mart_indicadores_financeiros_todas_empresas.sql
Banco alvo: localhost:5433/bigdata_for_finance


## 2. Ler a query


In [2]:
sql_text = SQL_PATH.read_text(encoding='utf-8')
print(f'Tamanho da query: {len(sql_text):,} caracteres')
print(sql_text[:1200])


Tamanho da query: 10,175 caracteres
-- Recria a mart no mesmo formato da versao anterior, agora para todas as empresas.
-- Ajustes principais:
-- 1. Remove o filtro de setor.
-- 2. Corrige AT_BIOLOGICO para CD_CONTA = '1.01.05'.
-- 3. Usa AT_BIOLOGICO apenas quando existir, sem penalizar empresas de outros setores.
-- 4. Ajusta LIQ_IMEDIATA para considerar Caixa + Aplicacoes Financeiras.
-- 5. Ajusta PMRE e Giro de Estoques para considerar Estoques + Ativos Biologicos quando houver.
-- 6. Completa todos os indicadores formula-definidos em Indicadores/00_racional_indicadores/indicadores_financeiros.md.

DROP TABLE IF EXISTS "layer_03_gold"."mart_indicadores_financeiros";

CREATE TABLE "layer_03_gold"."mart_indicadores_financeiros" AS
WITH deduplicado_bp AS (
    SELECT *
    FROM (
        SELECT
            "CNPJ_CIA",
            "DT_REFER",
            "DENOM_CIA",
            "SETOR_ATIV",
            "CD_CONTA",
            "VL_CONTA_TRATADO",
            "VERSAO",
            ROW_

## 3. Executar no PostgreSQL

A SQL contem mais de um comando (`DROP TABLE` + `CREATE TABLE AS ...`), entao a execucao usa a conexao bruta do driver.


In [3]:
raw_conn = engine.raw_connection()
try:
    with raw_conn.cursor() as cur:
        cur.execute(sql_text)
    raw_conn.commit()
    print('OK - query executada com sucesso em layer_03_gold.mart_indicadores_financeiros')
finally:
    raw_conn.close()


OK - query executada com sucesso em layer_03_gold.mart_indicadores_financeiros


## 4. Confirmacao rapida


In [4]:
q_count = '''
SELECT COUNT(*) AS n_linhas
FROM "layer_03_gold"."mart_indicadores_financeiros";
'''

q_amostra = '''
SELECT *
FROM "layer_03_gold"."mart_indicadores_financeiros"
ORDER BY "CNPJ_CIA", "DT_REFER"
LIMIT 10;
'''

import pandas as pd

df_count = pd.read_sql(text(q_count), engine)
df_sample = pd.read_sql(text(q_amostra), engine)

display(df_count)
display(df_sample)


,n_linhas
0,2221


,CNPJ_CIA,DT_REFER,RAZAO_SOCIAL,SETOR,TP_MERC,AT,AC,CAIXA_EQUIV,APLIC_FIN,CLIENTES,...,GIRO_AC,NCG,ST,PMRV,PMRE,PMPC,PMRAC,CICLO_ECONOMICO,CICLO_FINANCEIRO,EFEITO_TESOURA
0,00.070.698/0001-11,2010-12-31,COMPANHIA ENERGÉTICA DE BRASÍLIA - CEB,Energia Elétrica,MERCADO_ABERTO,2.119934e+09,4.499600e+08,99258000.0,NaN,321170000.0,...,4.087741,-123045000.0,-54941000.0,62.860840,4.823122,80.014363,88.068199,67.683962,-12.330401,False
1,00.070.698/0001-11,2011-12-31,COMPANHIA ENERGÉTICA DE BRASÍLIA - CEB,Energia Elétrica,MERCADO_ABERTO,2.170285e+09,4.572840e+08,66748000.0,NaN,357186000.0,...,3.012611,-139630000.0,-60851000.0,93.340002,3.065170,52.313517,119.497655,96.405172,44.091655,False
2,00.070.698/0001-11,2012-12-31,COMPANHIA ENERGÉTICA DE BRASÍLIA - CEB,Energia Elétrica,MERCADO_ABERTO,2.424419e+09,5.705350e+08,185433000.0,9805000.0,308111000.0,...,2.854650,-133982000.0,89225000.0,68.104291,2.359371,46.573824,126.110011,70.463662,23.889838,False
3,00.070.698/0001-11,2013-12-31,COMPANHIA ENERGÉTICA DE BRASÍLIA - CEB,Energia Elétrica,MERCADO_ABERTO,2.434831e+09,5.208020e+08,96786000.0,295000.0,308840000.0,...,3.088838,-354374000.0,-22126000.0,69.114357,8.964314,89.112503,116.548683,78.078671,-11.033833,False
4,00.070.698/0001-11,2014-12-31,COMPANHIA ENERGÉTICA DE BRASÍLIA - CEB,Energia Elétrica,MERCADO_ABERTO,2.709827e+09,7.794110e+08,66006000.0,NaN,441174000.0,...,2.723573,-164177000.0,-3411000.0,74.818123,3.848033,77.714648,132.179295,78.666156,0.951508,False
5,00.070.698/0001-11,2015-12-31,COMPANHIA ENERGÉTICA DE BRASÍLIA - CEB,Energia Elétrica,MERCADO_ABERTO,3.344728e+09,1.556221e+09,78043000.0,NaN,548842000.0,...,1.559678,174565000.0,6150000.0,81.403624,1.309551,58.751931,230.816936,82.713175,23.961244,False
6,00.070.698/0001-11,2016-12-31,COMPANHIA ENERGÉTICA DE BRASÍLIA - CEB,Energia Elétrica,MERCADO_ABERTO,3.156892e+09,1.206344e+09,86041000.0,NaN,520706000.0,...,1.752862,-76043000.0,-10948000.0,88.649502,1.718813,42.969348,205.378458,90.368316,47.398967,False
7,00.070.698/0001-11,2017-12-31,COMPANHIA ENERGÉTICA DE BRASÍLIA - CEB,Energia Elétrica,MERCADO_ABERTO,3.637085e+09,1.695912e+09,92001000.0,NaN,555376000.0,...,1.604143,167779000.0,-97670000.0,73.492516,1.275204,65.597098,224.418843,74.767720,9.170621,False
8,00.070.698/0001-11,2018-12-31,COMPANHIA ENERGÉTICA DE BRASÍLIA - CEB,Energia Elétrica,MERCADO_ABERTO,3.741671e+09,1.842353e+09,179699000.0,NaN,622655000.0,...,1.405925,67658000.0,-154148000.0,86.539624,1.374416,79.499814,256.059192,87.914041,8.414227,True
9,00.070.698/0001-11,2019-12-31,COMPANHIA ENERGÉTICA DE BRASÍLIA - CEB,Energia Elétrica,MERCADO_ABERTO,3.687355e+09,1.740536e+09,465338000.0,NaN,608867000.0,...,1.579823,-145235000.0,317691000.0,79.713820,1.645876,31.322177,227.873696,81.359696,50.037518,False
